## Análisis geográfico y social de centros médicos en chile ##

**Proyecto infovis**

- Martin castro 
- victoria otarola
- josefa venegas

## introduccion ##

La distribución de la infraestructura sanitaria en una ciudad no solo refleja su desarrollo urbano, sino también las desigualdades sociales que atraviesan a sus habitantes. En este proyecto, realizamos un análisis geográfico y social de los establecimientos de salud vigentes en la Región Metropolitana de Chile, utilizando datos abiertos publicados por el Ministerio de Salud. A través de una visualización basada en mapas de calor, buscamos identificar patrones de concentración y carencias en la ubicación de hospitales, consultorios, clínicas y otros centros médicos. Nuestro objetivo es evidenciar las diferencias entre comunas urbanas y periféricas en el acceso a la atención en salud, promoviendo así una reflexión informada sobre la equidad territorial. Este trabajo fue desarrollado por Martin Castro, Victoria Otárola y Josefa Venegas.

In [10]:
import folium
from folium.plugins import HeatMap
import pandas as pd
import numpy as np
from PIL import Image
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy.util import column_set

In [11]:
df = pd.read_csv("Datos\establecimientos_20250415.csv",sep=';')

df = df.drop(columns=['EstablecimientoCodigoAntiguo', 'EstablecimientoCodigoMadreAntiguo',
       'EstablecimientoCodigoMadreNuevo','SeremiSaludCodigo_ServicioDeSaludCodigo',
       'SeremiSaludGlosa_ServicioDeSaludGlosa','RegionCodigo', 'SeremiSaludCodigo_ServicioDeSaludCodigo',
       'ComunaCodigo',])

df = df[df['RegionGlosa'] == 'Región Metropolitana de Santiago']
df = df[df['NivelComplejidadEstabGlosa'] != 'nan']
df = df[df['NivelComplejidadEstabGlosa'] != 'No Aplica']

df = df.reset_index()
df = df.dropna(subset=['Latitud','Longitud','NivelComplejidadEstabGlosa'])

<>:1: SyntaxWarning: invalid escape sequence '\e'
<>:1: SyntaxWarning: invalid escape sequence '\e'
C:\Users\Asus\AppData\Local\Temp\ipykernel_20492\4283084581.py:1: SyntaxWarning: invalid escape sequence '\e'
  df = pd.read_csv("Datos\establecimientos_20250415.csv",sep=';')


In [12]:
tipo_1 = ["cesfam","sapu","psr","cecosf","consultorio","cgr"]
tipo_2 = ["hospital","clínica","centro de salud"]

def tipo_salud(x):
    for y in tipo_1:
        if y in x.lower():
            return 'Prioritario'
    for y in tipo_2:
        if y in x.lower():
            return 'Secundario'
    return 'Otro'
            


df['Nivel_urgencia'] = df['TipoEstablecimientoGlosa'].apply(tipo_salud)

df['Nivel_urgencia'].value_counts()


Nivel_urgencia
Prioritario    380
Secundario     376
Otro           154
Name: count, dtype: int64

In [13]:
with open('Datos/poblacion.json', 'r') as f:
    poblacion = json.load(f)
print(poblacion['Región Metropolitana'])

df_cantidad = pd.DataFrame( df['ComunaGlosa'].value_counts().sort_index())
df_cantidad = df_cantidad.reset_index()
df_cantidad = df_cantidad.rename(columns={'count': 'Hospitales'})
df_cantidad = df_cantidad.rename(columns={'ComunaGlosa': 'Comuna'})

df_cantidad['personas'] = df_cantidad['Comuna'].apply(lambda x: poblacion['Región Metropolitana'][str(x)] )
df_cantidad['Prioritario'] = None 
df_cantidad['Secundario'] = None 
df_cantidad['Otro'] = None 

df_cantidad

{'Alhué': 5000, 'Buin': 100000, 'Calera de Tango': 25000, 'Cerrillos': 95000, 'Cerro Navia': 150000, 'Colina': 170000, 'Conchalí': 130000, 'Curacaví': 35000, 'El Bosque': 180000, 'El Monte': 35000, 'Estación Central': 150000, 'Huechuraba': 85000, 'Independencia': 95000, 'Isla de Maipo': 35000, 'Lampa': 130000, 'La Cisterna': 85000, 'La Florida': 380000, 'La Granja': 120000, 'La Pintana': 200000, 'La Reina': 95000, 'Las Condes': 300000, 'Lo Barnechea': 131053, 'Lo Espejo': 100694, 'Lo Prado': 102078, 'Macul': 138361, 'Maipú': 586812, 'Maria Pinto': 12000, 'Melipilla': 130000, 'Padre Hurtado': 50000, 'Paine': 80000, 'Pedro Aguirre Cerda': 104462, 'Peñaflor': 70000, 'Peñalolén': 272913, 'Pirque': 25000, 'Providencia': 164009, 'Pudahuel': 261596, 'Puente Alto': 700000, 'Quilicura': 275744, 'Quinta Normal': 141823, 'Recoleta': 196856, 'Renca': 163102, 'San Bernardo': 310000, 'San Joaquín': 103118, 'San José de Maipo': 15000, 'San Miguel': 107000, 'San Pedro': 8000, 'San Ramón': 95000, 'Sant

,Comuna,Hospitales,personas,Prioritario,Secundario,Otro
0,Alhué,6,5000,None,None,None
1,Buin,18,100000,None,None,None
2,Calera de Tango,4,25000,None,None,None
3,Cerrillos,9,95000,None,None,None
4,Cerro Navia,10,150000,None,None,None
5,Colina,16,170000,None,None,None
6,Conchalí,13,130000,None,None,None
7,Curacaví,3,35000,None,None,None
8,El Bosque,14,180000,None,None,None
9,El Monte,3,35000,None,None,None


In [14]:



columnas = ['ComunaGlosa','Nivel_urgencia']

df_2 = df[columnas] 

df_Prioritario = pd.DataFrame( df_2[df_2['Nivel_urgencia'] == 'Prioritario'].value_counts().sort_index())
df_Prioritario= df_Prioritario.reset_index()
df_Prioritario = df_Prioritario.rename(columns={'count': 'Prioritario'})
df_Prioritario = df_Prioritario.rename(columns={'ComunaGlosa': 'Comuna'})

df_Secundario = pd.DataFrame( df_2[df_2['Nivel_urgencia'] == 'Secundario'].value_counts().sort_index())
df_Secundario= df_Secundario.reset_index()
df_Secundario = df_Secundario.rename(columns={'count': 'Secundario'})
df_Secundario = df_Secundario.rename(columns={'ComunaGlosa': 'Comuna'})

df_Otro = pd.DataFrame( df_2[df_2['Nivel_urgencia'] == 'Otro'].value_counts().sort_index())
df_Otro= df_Otro.reset_index()
df_Otro = df_Otro.rename(columns={'count': 'Otro'})
df_Otro = df_Otro.rename(columns={'ComunaGlosa': 'Comuna'})

df_cantidad = pd.merge(df_cantidad, df_Prioritario, on='Comuna')
df_cantidad = pd.merge(df_cantidad, df_Secundario, on='Comuna')
df_cantidad = pd.merge(df_cantidad, df_Otro, on='Comuna')

df_cantidad.drop(columns= ['Nivel_urgencia_x','Nivel_urgencia_y','Nivel_urgencia','Prioritario_x','Secundario_x','Otro_x'], axis=1, inplace=True)

df_cantidad = df_cantidad.rename(columns={'Prioritario_y': 'Prioritario'})
df_cantidad = df_cantidad.rename(columns={'Otro_y': 'Otro'})
df_cantidad = df_cantidad.rename(columns={'Secundario_y': 'Secundario'})


## Funcion graficar comuna 

In [21]:
# Configuración visual
sns.set(style="whitegrid")
plt.rcParams.update({'font.size': 10})

def graficar_comuna(df, comuna, carpeta_salida="img"):
    os.makedirs(carpeta_salida, exist_ok=True)
    
    fila = df[df['Comuna'] == comuna].iloc[0]

    # Reagrupación en 3 categorías
    niveles = ['Primario', 'Secundario', 'Otro']
    primario = fila['Prioritario']
    secundario = fila['Secundario']
    otro = fila['Otro'] + (fila['Hospitales'] - fila['Prioritario'] - fila['Secundario'] - fila['Otro'])
    valores = [primario, secundario, otro]

    total_centros = sum(valores)
    poblacion = fila['personas']
    centros_por_100k = (total_centros / poblacion) * 100000
    porcentajes = [v / total_centros * 100 for v in valores]
    colores = ['#4CAF50', '#2196F3', '#FFC107']  # Verde, Azul, Amarillo

    # Gráfico
    fig, ax = plt.subplots(figsize=(10, 5))
    barras = sns.barplot(x=niveles, y=valores, palette=colores, ax=ax)

    for i, valor in enumerate(valores):
        ax.text(i, valor + 0.5, f'{valor}', ha='center', va='bottom', fontweight='bold')

    ax.set_title(f'Centros Médicos en {comuna}', fontsize=14, fontweight='bold')
    ax.set_ylabel("Cantidad de Centros")
    ax.set_xlabel("Tipo de Atencion")

    texto = (
        f"Población: {poblacion:,}\n"
        f"Total centros médicos: {total_centros}\n"
        f"Centros por 100,000 habitantes: {centros_por_100k:.2f}\n\n"
        f"Distribución porcentual:\n"
        + "\n".join([f"{n}: {p:.1f}%" for n, p in zip(niveles, porcentajes)])
    )

    plt.text(3.1, max(valores), texto, va='top', fontsize=10, family='monospace')
    plt.tight_layout()

    # Guardar gráfico
    nombre_archivo = os.path.join(carpeta_salida, f"{comuna.replace(' ', '_')}.png")
    plt.savefig(nombre_archivo, dpi=300, bbox_inches='tight')
    plt.close()

In [22]:
for x in df_cantidad['Comuna']:
    graficar_comuna(df_cantidad,x)

C:\Users\Asus\AppData\Local\Temp\ipykernel_20492\1826779063.py:25: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  barras = sns.barplot(x=niveles, y=valores, palette=colores, ax=ax)
C:\Users\Asus\AppData\Local\Temp\ipykernel_20492\1826779063.py:25: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  barras = sns.barplot(x=niveles, y=valores, palette=colores, ax=ax)
C:\Users\Asus\AppData\Local\Temp\ipykernel_20492\1826779063.py:25: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  barras = sns.barplot(x=niveles, y=valores, palette=colores, ax=ax)
C:\Users\Asus\AppData\Local\Temp\ipykernel_

In [17]:
# Asegúrate de que tienes los datos limpios (sin valores NaN)
hola = df[df['RegionGlosa'] == 'Región Metropolitana de Santiago']
hola.reset_index(inplace=True)
hola_limpio = hola.dropna(subset=['Latitud','Longitud'])

# Importar las bibliotecas necesarias
import folium
import json

# Cargar el archivo GeoJSON de Chile
# Asegúrate de que la ruta sea correcta según donde hayas guardado el archivo
with open('Datos\\13.geojson', 'r', encoding='utf-8') as f:
    chile_geojson = json.load(f)

# Crear un mapa centrado en Chile
mapa = folium.Map(location=[-35.6751, -71.5430], zoom_start=5)  # Coordenadas aproximadas del centro de Chile

# Añadir los límites de Chile al mapa
folium.GeoJson(
    chile_geojson,
    name='Chile',
    style_function=lambda x: {
        'color': 'black',
        'weight': 2,
        'fillOpacity': 0.1
    }
).add_to(mapa)

# Función para determinar el color según el nivel de complejidad
def obtener_color(NivelComplejidadEstabGlosa):
    if NivelComplejidadEstabGlosa == 'Alta Complejidad':
        return 'red'
    elif NivelComplejidadEstabGlosa	 == 'Mediana Complejidad':
        return 'orange'
    elif NivelComplejidadEstabGlosa	 == 'Baja Complejidad':
        return 'green'

# Añadir círculos pequeños para cada punto con color según nivel de complejidad
for idx, fila in hola_limpio.iterrows():
    # Obtener el color según el nivel de complejidad
    color = obtener_color(fila.get('NivelComplejidadEstabGlosa', ''))
    
    folium.CircleMarker(
        location=[fila['Latitud'], fila['Longitud']],
        radius=1,  # Tamaño pequeño del círculo
        color=color,  # Color del borde según nivel de complejidad
        fill=True,
        fill_color=color,  # Color de relleno según nivel de complejidad
        fill_opacity=0.7,
        popup=f"Índice: {idx}<br>Nivel: {fila.get('NivelComplejidadEstabGlosa', 'No especificado')}",
        tooltip=f"Punto {idx}: {fila.get('NivelComplejidadEstabGlosa', 'No especificado')}"
    ).add_to(mapa)

# Añadir leyenda al mapa
leyenda_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 180px; height: 120px; 
            border:2px solid grey; z-index:9999; font-size:14px;
            background-color:white; padding: 10px;
            border-radius: 5px;">
    <p><strong>Nivel Complejidad</strong></p>
    <p><i class="fa fa-circle" style="color:red"></i> Alta Complejidad</p>
    <p><i class="fa fa-circle" style="color:orange"></i> Mediana Complejidad</p>
    <p><i class="fa fa-circle" style="color:green"></i> Baja Complejidad</p>
</div>
'''
mapa.get_root().html.add_child(folium.Element(leyenda_html))

# Limitar el mapa a los límites de Chile
# Esto ajustará automáticamente el zoom para mostrar todo Chile
mapa.fit_bounds(folium.GeoJson(chile_geojson).get_bounds())

# Añadir control de capas para poder activar/desactivar los límites
folium.LayerControl().add_to(mapa)

# Guardar el mapa como archivo HTML
mapa.save('mapa_chile_con_puntos_coloreados.html')

# Mostrar el mapa en el notebook
mapa

Nuestra visualización muestra la distribución de los distintos tipos de establecimientos de salud vigentes en la Región Metropolitana, como hospitales, consultorios, clínicas y centros médicos. El objetivo es analizar cómo se reparte la infraestructura sanitaria dentro de esta región, evidenciando diferencias entre comunas urbanas y periféricas. Buscamos destacar posibles desigualdades en el acceso a la atención según el tipo de establecimiento disponible en cada zona.

## Gráficos de Centros Médicos por Comuna en la Región Metropolitana ##

Se realizó un análisis de datos utilizando Python, donde se generaron gráficos que muestran la cantidad de centros médicos en la Región Metropolitana y su distribución según nivel de complejidad. Para este análisis, se elaboró un gráfico por comuna, permitiendo visualizar la realidad sanitaria de cada territorio de forma individual.

Actualmente, los gráficos se presentan de manera secuencial, uno debajo del otro. Sin embargo, se proyecta que en futuras versiones la visualización sea más interactiva, permitiendo desplazarse horizontalmente entre los gráficos de cada comuna o acceder a ellos mediante una galería dinámica, mejorando así la experiencia de exploración y comparación de datos.



**Nota sobre los datos poblacionales:**

Las cifras de población comunal utilizadas en este análisis fueron obtenidas desde el portal de la Biblioteca del Congreso Nacional de Chile (BCN), a través de su sección de reportes comunales: https://www.bcn.cl/siit/reportescomunales/comunas_v.html?anno=2024&idcom=13201. Estos datos corresponden a estimaciones para el año 2024 y constituyen aproximaciones que pueden estar sujetas a futuras actualizaciones y cambios demográficos, según las proyecciones oficiales y censos nacionales.



## datos

Los datos fueron extraídos de "https://datos.gob.cl/dataset/establecimientos-de-salud-vigentes", trabajados de tal forma que datos nulos o poco reprensivos fueran borrados y filtrados según características que favorecieran nuestro análisis 

## biblografia ## 

https://lineasdebasepublicas.mma.gob.cl/datos_abiertos/dataset/limite-urbano-prms-metropolitana

https://github.com/caracena/chile-geojson

https://datos.gob.cl/dataset/establecimientos-de-salud-vigentes

https://www.bcn.cl/siit/reportescomunales/comunas_v.html?anno=2024&idcom=13201



In [18]:
import os

# Carpeta a recorrer
ruta_carpeta = 'img'

# Listar todos los nombres de archivos
archivos = os.listdir(ruta_carpeta)

# Filtrar solo archivos (opcional)
archivos = [f for f in archivos if os.path.isfile(os.path.join(ruta_carpeta, f))]

print(archivos)


['Buin.png', 'Cerrillos.png', 'Cerro_Navia.png', 'Colina.png', 'Conchalí.png', 'El_Bosque.png', 'Estación_Central.png', 'Huechuraba.png', 'Independencia.png', 'Las_Condes.png', 'La_Florida.png', 'La_Pintana.png', 'La_Reina.png', 'Lo_Barnechea.png', 'Lo_Espejo.png', 'Macul.png', 'Maipú.png', 'Melipilla.png', 'Paine.png', 'Pedro_Aguirre_Cerda.png', 'Peñaflor.png', 'Peñalolén.png', 'Providencia.png', 'Pudahuel.png', 'Puente_Alto.png', 'Quilicura.png', 'Quinta_Normal.png', 'Recoleta.png', 'Santiago.png', 'San_Bernardo.png', 'San_José_de_Maipo.png', 'San_Miguel.png', 'San_Ramón.png', 'Talagante.png', 'Tiltil.png', 'Vitacura.png', 'Ñuñoa.png']
